In [ ]:
from datasets import load_dataset

dump = load_dataset("heegyu/namuwiki-sentences", split="train")
print(len(dump))

In [ ]:
import json
from collections import deque

with open('word_to_idx.json', 'r') as f:
    word_to_idx = json.load(f)
with open('idx_to_word.json', 'r') as f:
    idx_to_word = json.load(f)

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.fail = None
        self.output = []


def build_trie(patterns):
    root = TrieNode()
    for pat in patterns:
        node = root
        for ch in pat:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.output.append(pat)
    return root


def build_failure_links(root):
    queue = deque()
    for child in root.children.values():
        child.fail = root
        queue.append(child)
    root.fail = None

    while queue:
        node = queue.popleft()
        for ch, child in node.children.items():
            queue.append(child)
            fail_node = node.fail
            while fail_node and ch not in fail_node.children:
                fail_node = fail_node.fail
            if fail_node:
                child.fail = fail_node.children[ch]
            else:
                child.fail = root
            child.output += child.fail.output


def aho_corasick_search(text, root, patterns):
    result = []
    node = root
    for ch in text:
        if ch == ' ':
            continue

        while node is not root and ch not in node.children:
            node = node.fail
        if ch in node.children:
            node = node.children[ch]
        else:
            node = root
        for pat in node.output:
            result.append(word_to_idx[pat])
    return result

In [ ]:
keywords = list(word_to_idx.keys())
root = build_trie(keywords)
build_failure_links(root)

inverse_index = {keyword : set() for keyword in range(len(idx_to_word))}

In [ ]:
for i in range(len(dump)):
    row = dump[i]
    title, sentence = row['title'].replace(" ", ""), row['sentence']
    if len(title) + len(sentence) > 640:
        continue
    match = aho_corasick_search(sentence, root, keywords)
    if title in keywords:
        match.append(word_to_idx[title])
    if match:
        for m in match:
            inverse_index[m].add(i)

    if i % 1000000 == 0:
        print(i)

In [ ]:
for key in inverse_index:
    inverse_index[key] = list(inverse_index[key])

with open("inverse_index.json", "w") as f:
    json.dump(inverse_index, f, indent=4)